# Multi-target curvature training and evaluation notebook

Set `PREDICTION_MODE` to switch between:

- `gaussian_only`: first target only, Gaussian curvature
- `mean_only`: second target only, mean curvature
- `both_diagonal`: both targets with independent Gaussian output variances
- `both_full`: both targets with a full covariance Gaussian output

For plots that need a scalar quantity, set `PLOT_TARGET_INDEX_WITHIN_SELECTED`. In both-target modes, `0` plots Gaussian curvature and `1` plots mean curvature.


In [ ]:
import copy
import inspect
import json
import math
import pickle
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "training_data"

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())


In [ ]:
# Data and target settings
DATASET_NAME = "mean_curvature_smooth"
USE_GLOBAL_FEATURES = True
PREDICTION_MODE = "gaussian_only"  # "gaussian_only", "mean_only", "both_diagonal", "both_full"
MODE_CONFIGS = {
    "gaussian_only": {"target_indices": [0], "covariance_mode": "diagonal"},
    "mean_only": {"target_indices": [1], "covariance_mode": "diagonal"},
    "both_diagonal": {"target_indices": [0, 1], "covariance_mode": "diagonal"},
    "both_full": {"target_indices": [0, 1], "covariance_mode": "full"},
}
TARGET_LABELS_ALL = {
    0: "Gaussian curvature",
    1: "Mean curvature",
}
if PREDICTION_MODE not in MODE_CONFIGS:
    raise ValueError(f"Unknown PREDICTION_MODE={PREDICTION_MODE!r}")
TARGET_INDICES = list(MODE_CONFIGS[PREDICTION_MODE]["target_indices"])
TARGET_DIM = len(TARGET_INDICES)
COVARIANCE_MODE = MODE_CONFIGS[PREDICTION_MODE]["covariance_mode"]
TARGET_LABELS = [TARGET_LABELS_ALL.get(i, f"target {i}") for i in TARGET_INDICES]
PLOT_TARGET_INDEX_WITHIN_SELECTED = 0
PLOT_TARGET_LABEL = TARGET_LABELS[PLOT_TARGET_INDEX_WITHIN_SELECTED]
PLOT_ALL_TARGETS = [(i, TARGET_LABELS[i]) for i in range(TARGET_DIM)]

# Filtering and preprocessing settings
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Split and sampling settings
VAL_FRAC = 0.1
SPLIT_SEED = None
FORCED_VAL_KEYS = {
    ("20251201", "day4p5_B03_144"),
}

# Model settings
NUM_LAYERS = 4
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True

# Training settings
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 2000
PATIENCE = 30
NUM_WORKERS = 4
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

print("PREDICTION_MODE =", PREDICTION_MODE)
print("TARGET_INDICES  =", TARGET_INDICES)
print("TARGET_DIM      =", TARGET_DIM)
print("COVARIANCE_MODE =", COVARIANCE_MODE)
print("Plot target     =", PLOT_TARGET_LABEL)


In [ ]:
# =============================================================================
# MULTI-TARGET HELPERS
# =============================================================================
from src.data.io import select_graph_targets


def as_2d_targets(a):
    a = np.asarray(a)
    return a[:, None] if a.ndim == 1 else a


def select_target_array(a, target_index=0):
    a2 = as_2d_targets(a)
    if target_index >= a2.shape[1]:
        raise IndexError(f"target_index={target_index}, array shape={a2.shape}")
    return a2[:, int(target_index)]


def prepare_scalar_prediction_arrays(y, mu, log_var=None, target_index=0):
    y_s = select_target_array(y, target_index)
    mu_s = select_target_array(mu, target_index)
    if log_var is None:
        return y_s, mu_s, None
    lv_s = select_target_array(log_var, target_index)
    return y_s, mu_s, lv_s


def safe_sem(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return np.nan if values.size <= 1 else float(np.std(values, ddof=1) / np.sqrt(values.size))


def finite_mae(y, mu):
    y, mu = np.asarray(y), np.asarray(mu)
    ok = np.isfinite(y) & np.isfinite(mu)
    return np.nan if not np.any(ok) else float(np.mean(np.abs(mu[ok] - y[ok])))


def finite_mse(y, mu):
    y, mu = np.asarray(y), np.asarray(mu)
    ok = np.isfinite(y) & np.isfinite(mu)
    return np.nan if not np.any(ok) else float(np.mean((mu[ok] - y[ok]) ** 2))


def extract_node_array_for_selected_graphs(source_graphs, selected_graphs, values, *, key_attr="organoid_str"):
    value_map = {}
    offset = 0
    for g in source_graphs:
        n = int(g.y.shape[0])
        value_map[getattr(g, key_attr)] = np.asarray(values[offset:offset+n])
        offset += n

    out = []
    for g in selected_graphs:
        out.append(value_map[getattr(g, key_attr)])
    return np.concatenate(out, axis=0)


def iter_plot_targets(which="current"):
    """
    Helper to iterate over targets for plotting.

    Parameters
    ----------
    which : "current" | "all" | list[int]
        - "current": use PLOT_TARGET_INDEX_WITHIN_SELECTED
        - "all":     loop over all selected targets (TARGET_INDICES)
        - list:      explicit indices within selected targets

    Returns
    -------
    list of (target_index_within_selected, label)
    """

    if which == "current":
        return [(PLOT_TARGET_INDEX_WITHIN_SELECTED, PLOT_TARGET_LABEL)]

    if which == "all":
        return [(i, TARGET_LABELS[i]) for i in range(TARGET_DIM)]

    if isinstance(which, (list, tuple)):
        return [(i, TARGET_LABELS[i]) for i in which]

    raise ValueError("which must be 'current', 'all', or a list of indices")

In [ ]:
import matplotlib.pyplot as plt

def plot_train_val_distribution(
    g_train,
    g_val,
    *,
    target_idx=0,
    bins=100,
    density=True,
    alpha=0.5,
    logy=False,
    clip_quantiles=(0.01, 0.99),   # <-- key addition
):
    """
    Plot overlaid histograms (PDFs) for train vs validation targets,
    with outlier clipping for visualization (but reporting them).
    """

    def extract_targets(graphs):
        ys = []
        for g in graphs:
            y = g.y.detach().cpu().numpy()
            if y.ndim == 2:
                y = y[:, target_idx]
            ys.append(y.reshape(-1))
        return np.concatenate(ys)

    y_train = extract_targets(g_train)
    y_val   = extract_targets(g_val)

    # Remove NaNs/infs
    y_train = y_train[np.isfinite(y_train)]
    y_val   = y_val[np.isfinite(y_val)]

    # --- Compute clipping range from combined data ---
    y_all = np.concatenate([y_train, y_val])
    q_low, q_high = np.quantile(y_all, clip_quantiles)

    # --- Count outliers ---
    train_out_low  = np.sum(y_train < q_low)
    train_out_high = np.sum(y_train > q_high)
    val_out_low    = np.sum(y_val < q_low)
    val_out_high   = np.sum(y_val > q_high)

    # --- Clip for plotting ---
    y_train_clip = np.clip(y_train, q_low, q_high)
    y_val_clip   = np.clip(y_val, q_low, q_high)

    # --- Plot ---
    plt.figure(figsize=(6, 4))

    plt.hist(
        y_train_clip,
        bins=bins,
        range=(q_low, q_high),
        density=density,
        alpha=alpha,
        label="train",
    )

    plt.hist(
        y_val_clip,
        bins=bins,
        range=(q_low, q_high),
        density=density,
        alpha=alpha,
        label="val",
    )

    plt.xlabel("target value (clipped range)")
    plt.ylabel("density" if density else "count")
    plt.title("Train vs Validation Target Distribution (clipped)")
    plt.legend()

    if logy:
        plt.yscale("log")

    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    # --- Print stats ---
    print("\n=== Raw statistics ===")
    print(f"Train: mean={y_train.mean():.4f}, std={y_train.std():.4f}")
    print(f"Val  : mean={y_val.mean():.4f}, std={y_val.std():.4f}")

    print("\n=== Clipping info ===")
    print(f"Clip range: [{q_low:.4f}, {q_high:.4f}] (quantiles {clip_quantiles})")

    print("\nTrain outliers:")
    print(f"  below: {train_out_low} ({train_out_low/len(y_train):.2%})")
    print(f"  above: {train_out_high} ({train_out_high/len(y_train):.2%})")

    print("\nVal outliers:")
    print(f"  below: {val_out_low} ({val_out_low/len(y_val):.2%})")
    print(f"  above: {val_out_high} ({val_out_high/len(y_val):.2%})")

In [ ]:
from src.data.io import load_graph_dataset_from_dir
from src.data.metadata import load_aux_metadata_for_dir, attach_metadata_to_graphs
from src.data.metadata import load_marker_names_from_dir, print_graph_and_metadata_fields

data_dir = DATA_ROOT / DATASET_NAME

graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")
if graphs:
    print("Raw y shape:", tuple(graphs[0].y.shape))

meta = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, meta)

marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    n_markers = int(graphs[0].x.size(1)) if graphs else 0
    marker_names = [f"marker_{i}" for i in range(n_markers)]
print(f"Loaded {len(marker_names)} markers.")

graphs = select_graph_targets(graphs, TARGET_INDICES, inplace=False)
if graphs:
    print("Selected y shape:", tuple(graphs[0].y.shape))
    print("Selected targets:", TARGET_LABELS)

print_graph_and_metadata_fields(graphs)


In [ ]:
# Optional day3p5 timepoint filter
from src.data.filters import filter_graphs_by_metadata

DAY3P5_TIMEPOINT = "day3p5"

# Default: remove day3p5 graphs if present.
graphs = filter_graphs_by_metadata(
    graphs,
    key="timepoint",
    drop_values={DAY3P5_TIMEPOINT},
    missing="keep",
    inplace=False,
    print_summary=True,
)

# Alternative: keep only day3p5 graphs and filter out all other timepoints.
# Uncomment this block and comment out the default block above if needed.
# graphs = filter_graphs_by_metadata(
#     graphs,
#     key="timepoint",
#     keep_values={DAY3P5_TIMEPOINT},
#     missing="drop",
#     inplace=False,
#     print_summary=True,
# )


In [ ]:
from src.data.metadata import fill_missing_metadata_for_group
from src.data.filters import (
    filter_graphs_by_marker_diversity,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

graphs = fill_missing_metadata_for_group(
    graphs,
    field="complexity",
    fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
    dataset=MISSING_COMPLEXITY_GROUP["dataset"],
    timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
)

graphs, g_spherical = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
)

g_spherical = filter_graphs_by_marker_diversity(
    g_spherical,
    min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
    print_summary=True,
)

graphs = filter_graphs_by_numeric_metadata(
    graphs,
    key="complexity",
    min_value=COMPLEXITY_MIN,
    allow_missing=False,
    inplace=False,
    print_summary=True,
)

graphs = graphs + g_spherical
print(f"After filtering and spherical rescue: {len(graphs)} organoids.")

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        target_indices=None,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    outlier_info = None


In [ ]:
from src.data.metadata import add_log_metadata_features, promote_metadata_to_graph_tensors

field_specs = []

if USE_GLOBAL_FEATURES:
    graphs = add_log_metadata_features(graphs, inplace=False)
    field_specs = [
        {
            "meta_keys": [
                "log_surface_area",
                "log_volume",
                "log_volume_over_area",
                "log_num_cells",
            ],
            "attr_name": "global_feat",
            "kind": "graph_vector",
            "dtype": torch.float32,
        },
    ]
    graphs = promote_metadata_to_graph_tensors(graphs, field_specs, inplace=False)
    print("Promoted metadata fields to graph tensor attributes.")
else:
    print("Global features disabled; no global_feat attribute was attached.")


In [ ]:
from src.data.splits import train_val_split_graphs, graph_metadata_key

g_train, g_val, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    force_val_keys=FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
)

print(f"Split -> train: {len(g_train)} | val: {len(g_val)}")


In [ ]:
plot_train_val_distribution(g_train, g_val, target_idx=0, clip_quantiles=(0.001, 0.999))

In [ ]:
from src.data.metadata import strip_graph_metadata, snapshot_graph_metadata
from src.data.target_transforms import AsinhStandardizeTransform, standardize_graph_global_features

val_meta_lookup = snapshot_graph_metadata(g_val)

g_train = strip_graph_metadata(g_train, inplace=False)
g_val   = strip_graph_metadata(g_val, inplace=False)


# Fit on train only, then transform train/val targets in-place
target_transform = AsinhStandardizeTransform(robust=True).fit(g_train)
target_transform.transform_graphs(g_train)
target_transform.transform_graphs(g_val)

center_global, scale_global = None, None
if USE_GLOBAL_FEATURES:
    center_global, scale_global = standardize_graph_global_features(
        g_train,
        g_val,
        attr_name="global_feat",
        robust=False,
    )


In [ ]:
plot_train_val_distribution(g_train, g_val, target_idx=0, clip_quantiles=(0.0, 1.0))

In [ ]:
from src.models.gnn import GINCurvature
from src.data.metadata import infer_global_dim

num_layers = NUM_LAYERS
n_markers = int(g_train[0].x.size(1))
global_dim = infer_global_dim(g_train)

model = GINCurvature(
    n_markers=n_markers,
    global_dim=global_dim,
    hidden_dim=HIDDEN_DIM,
    num_layers=num_layers,
    dropout=DROPOUT,
    residual=RESIDUAL,
    norm=NORM,
    target_dim=TARGET_DIM,
    covariance_mode=COVARIANCE_MODE,
)

In [ ]:
from src.training.loop import TrainConfig, train
from src.training.losses import *

aux_losses = [
    WeightedLossTerm(
        name="edge",
        fn=edge_loss_term,
        weight=0.20,
        params={
            "weighted": False,
            "alpha": 2.0,
            "normalize_by": "graph_std",
            "clip_weight": 4.0,
        },
    ),
]

cfg = TrainConfig(
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    aux_losses=aux_losses,
)

model, metrics, history = train(model, g_train, g_val, cfg)


In [ ]:
plt.figure()
plt.plot(history.get("train_loss", []), label="train loss")
plt.plot(history.get("val_loss", []), label="val loss")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


In [ ]:
from src.inference.predict import predict_targets
from src.analysis.marker_stats import compute_markerwise_residuals
from src.data.metadata import load_marker_names_from_dir

device = cfg.device if "cfg" in globals() else ("cuda" if torch.cuda.is_available() else "cpu")

y_all, mu_all, log_var_all, X = predict_targets(
    g_val,
    model,
    device=device,
    return_log_var=True,
    target_transform=target_transform,
)

# These are scalar arrays used by target-agnostic plotting cells.
y_plot, mu_plot, log_var_plot = prepare_scalar_prediction_arrays(
    y_all,
    mu_all,
    log_var_all,
    target_index=PLOT_TARGET_INDEX_WITHIN_SELECTED,
)

print("y_all shape       :", np.shape(y_all))
print("mu_all shape      :", np.shape(mu_all))
print("log_var_all shape :", np.shape(log_var_all))
print("plot target       :", PLOT_TARGET_LABEL)

residuals_model, residuals_base, n_pos, mu_pos = compute_markerwise_residuals(y_plot, mu_plot, X)

marker_names = load_marker_names_from_dir(data_dir)
order_by_n = np.argsort(-n_pos)
marker_idx_show = order_by_n[:min(12, (X.shape[1] if X.ndim == 2 else 12))]
marker_names_sel = [marker_names[m] if m < len(marker_names) else f"m{m}" for m in marker_idx_show]


In [ ]:
from matplotlib.patches import Ellipse
from matplotlib.colors import LogNorm

def add_cov_ellipse(ax, x, y, n_std=1.0, **kwargs):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 2 or y.size < 2:
        return
    cov = np.cov(x, y)
    if not np.all(np.isfinite(cov)):
        return

    vals, vecs = np.linalg.eigh(cov)
    vals = np.maximum(vals, 0.0)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]
    theta = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    width, height = 2.0 * n_std * np.sqrt(vals)

    ell = Ellipse(
        xy=(x.mean(), y.mean()),
        width=width,
        height=height,
        angle=theta,
        fill=False,
        **kwargs,
    )
    ax.add_patch(ell)


for plot_target_index, plot_target_label in iter_plot_targets("all"):

    y_plot_t, mu_plot_t, log_var_plot_t = prepare_scalar_prediction_arrays(
        y_all,
        mu_all,
        log_var_all,
        target_index=plot_target_index,
    )

    residuals_model_t, residuals_base_t, n_pos_t, mu_pos_t = compute_markerwise_residuals(
        y_plot_t,
        mu_plot_t,
        X,
    )

    cols = 4
    rows = int(np.ceil(len(marker_idx_show) / cols)) if len(marker_idx_show) else 1
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.6, rows * 4.2), squeeze=False)
    axes = axes.ravel()

    all_true, all_pred = [], []
    for m in marker_idx_show:
        mask = X[:, m] > 0.5
        yt = np.asarray(y_plot_t[mask], dtype=float)
        yp = np.asarray(mu_plot_t[mask], dtype=float)
        good = np.isfinite(yt) & np.isfinite(yp)
        if np.any(good):
            all_true.append(yt[good])
            all_pred.append(yp[good])

    if not all_true:
        raise RuntimeError(f"No valid points found for selected markers: {plot_target_label}")

    all_true = np.concatenate(all_true)
    all_pred = np.concatenate(all_pred)
    global_lo = min(all_true.min(), all_pred.min())
    global_hi = max(all_true.max(), all_pred.max())
    pad = 0.05 * (global_hi - global_lo + 1e-12)
    global_lo -= pad
    global_hi += pad

    nbins = 50
    hist_range = [[global_lo, global_hi], [global_lo, global_hi]]

    for k, m in enumerate(marker_idx_show):
        ax = axes[k]
        mask = X[:, m] > 0.5
        yt = np.asarray(y_plot_t[mask], dtype=float)
        yp = np.asarray(mu_plot_t[mask], dtype=float)
        good = np.isfinite(yt) & np.isfinite(yp)
        yt, yp = yt[good], yp[good]

        if yt.size == 0:
            ax.axis("off")
            continue

        n = yt.size
        mean_t, mean_p = yt.mean(), yp.mean()
        std_t, std_p = yt.std(ddof=0), yp.std(ddof=0)
        rmse = np.sqrt(np.mean((yp - yt) ** 2))
        bias = np.mean(yp - yt)
        r = np.corrcoef(yt, yp)[0, 1] if n > 1 else np.nan

        ax.hist2d(yt, yp, bins=nbins, range=hist_range, norm=LogNorm(), cmin=1)
        ax.plot([global_lo, global_hi], [global_lo, global_hi], linestyle="--", linewidth=1)
        ax.scatter([mean_t], [mean_p], s=60, marker="x", linewidths=2)
        ax.plot([mean_t - std_t, mean_t + std_t], [mean_p, mean_p], linewidth=2)
        ax.plot([mean_t, mean_t], [mean_p - std_p, mean_p + std_p], linewidth=2)
        add_cov_ellipse(ax, yt, yp, n_std=1.0, linewidth=2)
        add_cov_ellipse(ax, yt, yp, n_std=2.0, linewidth=1, alpha=0.7)

        ax.set_xlim(global_lo, global_hi)
        ax.set_ylim(global_lo, global_hi)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(True, alpha=0.2)
        ax.set_title(
            f"{marker_names_sel[k]} (n⁺={n_pos_t[m]})\n"
            f"r={r:.2f}  RMSE={rmse:.3f}  bias={bias:.3f}",
            fontsize=9,
        )
        ax.set_xlabel(f"true {plot_target_label}")
        ax.set_ylabel(f"pred {plot_target_label}")

    for k in range(len(marker_idx_show), len(axes)):
        axes[k].axis("off")

    fig.suptitle(
        f"Validation predictions per marker: {plot_target_label}\n"
        "Density map with identity line, mean±std cross, and covariance ellipses",
        y=1.02,
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# Residual histograms.
cols = 4
rows = int(np.ceil(len(marker_idx_show) / cols)) if len(marker_idx_show) else 1
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.6, rows * 3.0), squeeze=False)
axes = axes.ravel()

all_resid = np.concatenate([r for r in residuals_model if r.size > 0] + [r for r in residuals_base if r.size > 0])
rmax = np.nanpercentile(np.abs(all_resid), 99) if all_resid.size else 0.25
rmax = max(float(rmax), 1e-6)
edges = np.linspace(-rmax, rmax, 101)

for k, m in enumerate(marker_idx_show):
    ax = axes[k]
    rb = residuals_base[m]
    rm = residuals_model[m]
    if rb.size == 0:
        ax.axis("off")
        continue
    ax.hist(rb, bins=edges, density=True, alpha=0.4, label="baseline")
    ax.hist(rm, bins=edges, density=True, alpha=0.6, label="model")
    ax.set_title(f"{marker_names_sel[k]} (n⁺={n_pos[m]})", fontsize=10)
    ax.grid(True, alpha=0.2)

for k in range(len(marker_idx_show), len(axes)):
    axes[k].axis("off")
if len(marker_idx_show):
    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="upper right")
fig.suptitle(f"Validation residuals per marker — {PLOT_TARGET_LABEL}", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
for plot_target_index, plot_target_label in iter_plot_targets("all"):

    y_plot_t, mu_plot_t, log_var_plot_t = prepare_scalar_prediction_arrays(
        y_all,
        mu_all,
        log_var_all,
        target_index=plot_target_index,
    )

    residuals_model_t, residuals_base_t, n_pos_t, mu_pos_t = compute_markerwise_residuals(
        y_plot_t,
        mu_plot_t,
        X,
    )

    mse_model_t = np.array([np.mean(r ** 2) if r.size > 0 else np.nan for r in residuals_model_t])
    mse_base_t  = np.array([np.mean(r ** 2) if r.size > 0 else np.nan for r in residuals_base_t])
    sem_model_t = np.array([safe_sem(r ** 2) for r in residuals_model_t])
    sem_base_t  = np.array([safe_sem(r ** 2) for r in residuals_base_t])

    x = np.arange(len(marker_names))

    plt.figure(figsize=(max(8, 0.5 * len(marker_names)), 4))
    plt.errorbar(x, mse_base_t,  yerr=sem_base_t,  fmt="o", capsize=3, label="baseline")
    plt.errorbar(x, mse_model_t, yerr=sem_model_t, fmt="x", capsize=3, label="model")
    plt.xticks(x, marker_names, rotation=60, ha="right")
    plt.ylabel("MSE (positives only)")
    plt.title(f"Validation: per-marker MSE with SEM — {plot_target_label}")
    plt.grid(True, axis="y", alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
for plot_target_index, plot_target_label in iter_plot_targets("all"):

    y_plot_t, mu_plot_t, log_var_plot_t = prepare_scalar_prediction_arrays(
        y_all,
        mu_all,
        log_var_all,
        target_index=plot_target_index,
    )

    residuals_model_t, residuals_base_t, n_pos_t, mu_pos_t = compute_markerwise_residuals(
        y_plot_t,
        mu_plot_t,
        X,
    )

    mse_base_t = np.array([np.mean(r ** 2) if r.size > 0 else np.nan for r in residuals_base_t])
    sem_base_t = np.array([safe_sem(r ** 2) for r in residuals_base_t])

    Xb = X > 0.5
    var_node_t = np.exp(log_var_plot_t)

    var_model_t = np.array([
        np.mean(var_node_t[Xb[:, m]]) if np.any(Xb[:, m]) else np.nan
        for m in range(X.shape[1])
    ])
    sem_var_model_t = np.array([
        safe_sem(var_node_t[Xb[:, m]]) if np.any(Xb[:, m]) else np.nan
        for m in range(X.shape[1])
    ])

    var_base_t = mse_base_t
    sem_var_base_t = sem_base_t

    x = np.arange(len(marker_names))

    plt.figure(figsize=(max(8, 0.5 * len(marker_names)), 4))
    plt.errorbar(x, var_base_t,  yerr=sem_var_base_t,  fmt="o", capsize=3, label="baseline (empirical var)")
    plt.errorbar(x, var_model_t, yerr=sem_var_model_t, fmt="x", capsize=3, label="model (predicted marginal var)")
    plt.xticks(x, marker_names, rotation=60, ha="right")
    plt.ylabel("Variance (positives only)")
    plt.title(f"Validation: per-marker variance with SEM — {plot_target_label}")
    plt.grid(True, axis="y", alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from src.analysis.prediction_analysis import build_graph_prediction_dataframe
from src.plotting.mesh_plots import project_predictions_to_mesh, plot_projected_true_vs_pred

val_perf_df = build_graph_prediction_dataframe(
    g_val,
    y_plot,
    mu_plot,
    meta_lookup=val_meta_lookup,
)

X_best = 3
Y_worst = 3
best_df = val_perf_df.nsmallest(X_best, "graph_mse")
worst_df = val_perf_df.nlargest(Y_worst, "graph_mse")

for label, df in [("BEST", best_df), ("WORST", worst_df)]:
    for _, row in df.iterrows():
        result = project_predictions_to_mesh(
            int(row["graph_index"]),
            g_val,
            y_plot,
            mu_plot,
            meta_lookup=val_meta_lookup,
        )
        fig = plot_projected_true_vs_pred(
            result,
            title=f"{label} | {PLOT_TARGET_LABEL} | {result['organoid_str']} | MAE={row['graph_mae']:.4f}",
        )
        fig.show()


In [ ]:
from src.data.subgraphs import build_ego_subgraphs_for_dataset
from src.data.subgraph_sampling import sample_subgraphs_coverage, print_sampling_summary

num_hops = num_layers

val_subgraphs = build_ego_subgraphs_for_dataset(
    g_val,
    num_hops=num_hops,
    max_centers_per_graph=None,
    seed=0,
)

sampled_val_subgraphs, sample_info = sample_subgraphs_coverage(
    val_subgraphs,
    marker_names=marker_names,
    k_hops=num_layers,
    max_subgraphs=1000,
    min_center_count=40,
    min_pair_count=25,
    seed=0,
)

print_sampling_summary(sample_info, marker_names)


In [ ]:
pair_cov = sample_info["pair_covered"]   # (H, M, M)
H, M, _ = pair_cov.shape

fig, axes = plt.subplots(
    1,
    H,
    figsize=(H * (0.38 * M + 1.8), 0.38 * M + 2.2),
)
if H == 1:
    axes = [axes]

for hi, ax in enumerate(axes):
    im = ax.imshow(pair_cov[hi], vmin=0, vmax=np.max(pair_cov), cmap="viridis", aspect="auto")
    ax.set_title(f"hop {hi + 1}", fontsize=11)
    ax.set_xticks(range(M))
    ax.set_xticklabels(marker_names, rotation=60, ha="right", fontsize=9)
    ax.set_yticks(range(M))
    if hi == 0:
        ax.set_yticklabels(marker_names, fontsize=9)
        ax.set_ylabel("center marker")
    else:
        ax.set_yticklabels([])
    ax.set_xlabel("neighborhood marker")

fig.subplots_adjust(left=0.08, right=0.88, bottom=0.18, top=0.85, wspace=0.25)
cax = fig.add_axes([0.90, 0.22, 0.02, 0.55])
cbar = fig.colorbar(im, cax=cax)
cbar.set_label("sample count")
fig.suptitle("Sampling coverage per hop", fontsize=13)
plt.show()


In [ ]:
from src.analysis.perturbation import compute_perturbation_influence_maps
from src.plotting.influence_maps import plot_influence_heatmap, plot_influence_center_resolved

# res = compute_perturbation_influence_maps(
#     subgraphs=sampled_val_subgraphs,
#     model=model,
#     marker_names=marker_names,
#     k_hops=num_hops,
#     mode="single",
#     max_subgraphs=None,
#     batch_size=128,
# )

# def select_influence_target(arr, target_index=PLOT_TARGET_INDEX_WITHIN_SELECTED):
#     arr = np.asarray(arr)
#     # Supports scalar arrays as-is plus common multi-target conventions.
#     if arr.ndim >= 3 and arr.shape[0] == TARGET_DIM:
#         return arr[target_index]
#     if arr.ndim >= 3 and arr.shape[-1] == TARGET_DIM:
#         return arr[..., target_index]
#     return arr

# for key, title in [
#     ("delta_mu_total", "Δμ (perturbed - base)"),
#     ("delta_lv_total", "Δlogvar (perturbed - base)"),
#     ("delta_mu_abs_total", "|Δμ|"),
#     ("delta_lv_abs_total", "|Δlogvar|"),
# ]:
#     if key in res:
#         plot_influence_heatmap(
#             select_influence_target(res[key]),
#             res["hops"],
#             res["marker_names"],
#             title=f"{title}, {PLOT_TARGET_LABEL}, perturb=zero, k={num_hops}",
#         )

for plot_target_index, plot_target_label in iter_plot_targets("all"):

    res_t = compute_perturbation_influence_maps(
        subgraphs=sampled_val_subgraphs,
        model=model,
        marker_names=marker_names,
        k_hops=num_hops,
        mode="single",
        max_subgraphs=None,
        batch_size=128,
        target_index=plot_target_index,
    )

    if "delta_mu_cmarker" in res_t:
        plot_influence_center_resolved(
            res_t["delta_mu_cmarker"],
            res_t["hops"],
            res_t["marker_names"],
            title=f"Δμ per perturbed marker (X) conditioned on center marker (Y) — {plot_target_label}",
            sort_center=False,
            center_zero=True,
            cmap="RdBu_r",
        )
        plt.show()

    if "delta_lv_cmarker" in res_t:
        plot_influence_center_resolved(
            res_t["delta_lv_cmarker"],
            res_t["hops"],
            res_t["marker_names"],
            title=f"Δlogvar per perturbed marker (X) conditioned on center marker (Y) — {plot_target_label}",
            sort_center=False,
            center_zero=True,
            cmap="RdBu_r",
        )
        plt.show()


In [ ]:
from src.analysis.motif_clustering import run_embedding_clustering

extraction, clustering_result, summary = run_embedding_clustering(
    graphs=g_val,
    model=model,
    batch_size=128,
    center_only=False,
    embedding_variant="local",
    residualize_global=False,
    clustering="gmm",
    n_clusters=None,
    k_values=range(2, 11),
    covariance_type="full",
    standardize=True,
    pca_dim=None,
    seed=0,
    marker_names=marker_names,
)

Y_true_all, Y_pred_all, log_var_embed_all = target_transform.inverse_distribution(
    extraction.y_true,
    extraction.y_pred,
    log_var=extraction.log_var,
)

Y_true = select_target_array(Y_true_all, PLOT_TARGET_INDEX_WITHIN_SELECTED)
Y_pred = select_target_array(Y_pred_all, PLOT_TARGET_INDEX_WITHIN_SELECTED)
log_var = select_target_array(log_var_embed_all, PLOT_TARGET_INDEX_WITHIN_SELECTED) if log_var_embed_all is not None else None
labels = clustering_result.labels


In [ ]:
from sklearn.manifold import TSNE
from matplotlib.colors import BoundaryNorm

Z = clustering_result.embeddings_used

Z2 = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(Z)

K = labels.max() + 1
cmap = plt.get_cmap("tab10", K)
bounds = np.arange(K + 1) - 0.5
norm = BoundaryNorm(bounds, cmap.N)

plt.figure(figsize=(6, 5))
sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=labels, cmap=cmap, norm=norm, s=8)
cbar = plt.colorbar(sc, ticks=np.arange(K))
cbar.set_label("cluster")
cbar.set_ticklabels([f"C{k}" for k in range(K)])
plt.title("Embedding clusters")
plt.tight_layout()
plt.show()

for plot_target_index, plot_target_label in iter_plot_targets("all"):

    Y_true_t, Y_pred_t, log_var_t = prepare_scalar_prediction_arrays(
        Y_true_all,
        Y_pred_all,
        log_var_all,
        target_index=plot_target_index,
    )

    vmin_true, vmax_true = np.nanpercentile(Y_true_t, [1, 99])
    vmin_pred, vmax_pred = np.nanpercentile(Y_pred_t, [1, 99])

    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=Y_pred_t, s=8, vmin=vmin_pred, vmax=vmax_pred)
    plt.colorbar(sc, label=f"predicted {plot_target_label}")
    plt.title(f"Embedding colored by predicted {plot_target_label}")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 5))
    sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=Y_true_t, s=8, vmin=vmin_true, vmax=vmax_true)
    plt.colorbar(sc, label=f"true {plot_target_label}")
    plt.title(f"Embedding colored by true {plot_target_label}")
    plt.tight_layout()
    plt.show()

    if log_var_t is not None:
        plt.figure(figsize=(6, 5))
        sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=log_var_t, s=8)
        plt.colorbar(sc, label=f"predicted marginal log_var: {plot_target_label}")
        plt.title(f"Embedding colored by predicted log_var — {plot_target_label}")
        plt.tight_layout()
        plt.show()

In [ ]:
from src.analysis.marker_stats import append_none_marker_column
from src.analysis.cluster_analysis import (
    cluster_marker_means,
    cluster_marker_distribution,
    cluster_order_from_values,
)
from src.plotting.cluster_plots import (
    plot_cluster_marker_heatmap,
    plot_cluster_boxplots,
)

center_mask = extraction.local_node_index == 0
X_ext, marker_names_ext = append_none_marker_column(extraction.x_markers, marker_names)

marker_means_all = cluster_marker_means(X_ext, labels)
marker_means_center = cluster_marker_means(X_ext, labels, center_mask=center_mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
plot_cluster_marker_heatmap(marker_means_all, marker_names_ext, title="All nodes", ax=axes[0], show_colorbar=False)
plot_cluster_marker_heatmap(marker_means_center, marker_names_ext, title="Center nodes only", ax=axes[1], colorbar_label="fraction positive", show_colorbar=True)
plt.tight_layout()
plt.show()

dist_all = cluster_marker_distribution(X_ext, labels)
dist_center = cluster_marker_distribution(X_ext, labels, center_mask=center_mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
plot_cluster_marker_heatmap(dist_all, marker_names_ext, title="All nodes", ax=axes[0], show_colorbar=False, colorbar_label="fraction of marker-positive nodes in cluster")
plot_cluster_marker_heatmap(dist_center, marker_names_ext, title="Center nodes only", ax=axes[1], show_colorbar=True, colorbar_label="fraction of marker-positive nodes in cluster")
plt.tight_layout()
plt.show()


In [ ]:
cluster_order = cluster_order_from_values(labels, Y_pred, reducer=np.mean)

plot_cluster_boxplots(
    [Y_true, Y_pred],
    labels,
    data_labels=["true", "predicted"],
    colors=["lightblue", "lightgreen"],
    ylabel=PLOT_TARGET_LABEL,
    title=f"True vs prediction by cluster — {PLOT_TARGET_LABEL}",
    cluster_order=cluster_order,
)

if log_var is not None:
    plot_cluster_boxplots(
        [np.exp(log_var) / 2, (Y_true - Y_pred) ** 2],
        labels,
        data_labels=["variance/2", "squared residuals"],
        colors=["lightgreen", "lightblue"],
        ylabel="value",
        title=f"Prediction uncertainty by cluster — {PLOT_TARGET_LABEL}",
        cluster_order=cluster_order,
    )


In [ ]:
from src.plotting.cluster_plots import plot_cluster_metadata_boxplots

cluster_order = cluster_order_from_values(labels, Y_pred, reducer=np.mean)

get_min_node_val = lambda x, n_nodes: (
    lambda a: (
        np.full(n_nodes, np.nan)
        if a.size == 0 else
        a.astype(float)
        if a.ndim == 1 else
        np.min(a, axis=0).astype(float)
        if a.shape[1] >= n_nodes else
        np.min(a, axis=1).astype(float)
    )
)(np.asarray(x))

_, _, df_dcrypt = plot_cluster_metadata_boxplots(
    g_val,
    labels,
    fields="d_crypts_graph",
    meta_lookup=val_meta_lookup,
    transform=get_min_node_val,
    return_dataframe=True,
)

cluster_order_dcrypt = df_dcrypt.groupby("cluster")["value"].median().sort_values().index

plot_cluster_metadata_boxplots(
    g_val,
    labels,
    fields="d_crypts_graph",
    meta_lookup=val_meta_lookup,
    transform=get_min_node_val,
    cluster_order=cluster_order_dcrypt,
    ylabel="min distance to crypt",
    title="Distribution of min(d_crypts_graph) by cluster",
)
plt.axhline(0.0, linestyle="--")
plt.show()


In [ ]:
from src.analysis.cluster_analysis import build_binned_cluster_fraction_table

dcrypt_min = df_dcrypt["value"].to_numpy()
has_crypt = np.isfinite(dcrypt_min)

def dcrypt_to_bin(dmin, has_crypt):
    if (not has_crypt) or (not np.isfinite(dmin)):
        return "No crypt"
    elif dmin < 0.8:
        return "< 0.8"
    elif dmin <= 1.2:
        return "0.8 - 1.2"
    else:
        return "> 1.2"

bin_order = ["No crypt", "< 0.8", "0.8 - 1.2", "> 1.2"]

plot_table, cluster_order, df_dcrypt_bins = build_binned_cluster_fraction_table(
    labels=labels,
    values=dcrypt_min,
    valid_mask=has_crypt,
    bin_func=dcrypt_to_bin,
    bin_order=bin_order,
    cluster_order_by="median_valid_value",
    include_all=True,
)

fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(plot_table), dtype=float)
colors = {"No crypt": "lightgray", "< 0.8": "tab:blue", "0.8 - 1.2": "tab:orange", "> 1.2": "tab:green"}

for b in bin_order:
    vals = plot_table[b].values
    ax.bar(np.arange(len(plot_table)), vals, bottom=bottom, label=b, color=colors[b])
    bottom += vals

xticklabels = ["All"] + [f"C{k}" for k in cluster_order]
ax.set_xticks(np.arange(len(plot_table)))
ax.set_xticklabels(xticklabels, rotation=45)
ax.set_xlabel("group")
ax.set_ylabel("fraction of nodes")
ax.set_title("Crypt-distance categories by cluster")
ax.legend(title="min distance to crypt")
plt.tight_layout()
plt.show()


In [ ]:
show_outliers = False
median_style = dict(color="black", linewidth=2)
cluster_order = cluster_order_from_values(labels, Y_pred, reducer=np.mean)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

plot_cluster_metadata_boxplots(
    g_val,
    labels,
    fields="log_volume_over_area",
    meta_lookup=val_meta_lookup,
    transform=lambda x: np.exp(x),
    cluster_order=cluster_order,
    include_all=True,
    ylabel="volume / area",
    title="Volume / area",
    facecolors=["gainsboro"] + ["lightblue"] * len(cluster_order),
    show_outliers=show_outliers,
    ax=axes[0],
    boxplot_kwargs={"medianprops": median_style},
)

plot_cluster_metadata_boxplots(
    g_val,
    labels,
    fields="log_num_cells",
    meta_lookup=val_meta_lookup,
    transform=lambda x: np.exp(x),
    cluster_order=cluster_order,
    include_all=True,
    ylabel="number of cells",
    title="Number of cells",
    facecolors=["gainsboro"] + ["salmon"] * len(cluster_order),
    show_outliers=show_outliers,
    ax=axes[1],
    boxplot_kwargs={"medianprops": median_style},
)

plot_cluster_metadata_boxplots(
    g_val,
    labels,
    fields="complexity",
    meta_lookup=val_meta_lookup,
    cluster_order=cluster_order,
    include_all=True,
    ylabel="complexity",
    title="Complexity",
    facecolors=["gainsboro"] + ["lightgreen"] * len(cluster_order),
    show_outliers=show_outliers,
    ax=axes[2],
    boxplot_kwargs={"medianprops": median_style},
)

plt.tight_layout()
plt.show()


In [ ]:
from src.data.metadata import get_graph_metadata

rows = []
cursor = 0

for g in g_val:
    n_nodes = int(g.y.shape[0])
    graph_labels = labels[cursor:cursor + n_nodes]
    md = get_graph_metadata(g, meta_lookup=val_meta_lookup, strict=True)
    measurement = f"{md['dataset']} | {md['timepoint']}"
    rows.append(pd.DataFrame({"cluster": graph_labels, "measurement": [measurement] * n_nodes}))
    cursor += n_nodes

df_measure = pd.concat(rows, ignore_index=True)

cluster_order = cluster_order_from_values(labels, Y_pred, reducer=np.mean)
measurement_order = df_measure["measurement"].value_counts().index

frac_measurements_within_cluster = pd.crosstab(
    df_measure["cluster"],
    df_measure["measurement"],
    normalize="index",
).reindex(index=cluster_order, columns=measurement_order, fill_value=0.0)

frac_all = df_measure["measurement"].value_counts(normalize=True).reindex(measurement_order, fill_value=0.0)
plot_table = pd.concat([pd.DataFrame([frac_all.values], index=["All"], columns=measurement_order), frac_measurements_within_cluster], axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(plot_table), dtype=float)

for measurement in measurement_order:
    vals = plot_table[measurement].values
    ax.bar(np.arange(len(plot_table)), vals, bottom=bottom, label=measurement)
    bottom += vals

xticklabels = ["All"] + [f"C{k}" for k in cluster_order]
ax.set_xticks(np.arange(len(plot_table)))
ax.set_xticklabels(xticklabels, rotation=45)
ax.set_xlabel("group")
ax.set_ylabel("fraction of nodes")
ax.set_title("Measurement composition within each cluster")
ax.legend(title="measurement", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
from src.analysis.cluster_analysis import build_cluster_exemplar_subgraphs
from src.plotting.motif_plots import (
    DEFAULT_MARKER_COLORS,
    plot_cluster_exemplar_subgraphs_radial,
    plot_marker_color_legend,
)

cluster_exemplars = build_cluster_exemplar_subgraphs(
    graphs=g_val,
    extraction=extraction,
    clustering_result=clustering_result,
    top_k_per_cluster=4,
    num_hops=model.num_layers,
    require_assigned_label=True,
)

plot_marker_color_legend(marker_colors=DEFAULT_MARKER_COLORS)
plt.show()

for cluster_id in sorted(cluster_exemplars):
    plot_cluster_exemplar_subgraphs_radial(
        cluster_exemplars,
        cluster_id=cluster_id,
        marker_names=marker_names,
        marker_colors=DEFAULT_MARKER_COLORS,
    )
    plt.show()


In [ ]:
from src.data.splits import select_graphs_by_keys, graph_metadata_key
from src.plotting.mesh_plots import project_predictions_to_mesh, plot_projected_true_vs_pred

g_val_selected = select_graphs_by_keys(
    g_val,
    split_info["forced_val_keys"],
    key_fn=graph_metadata_key,
    meta_lookup=val_meta_lookup,
)

selected_meta_lookup = {
    getattr(g, "organoid_str"): val_meta_lookup[getattr(g, "organoid_str")]
    for g in g_val_selected
}

y_true_selected_all, y_pred_selected_all, log_var_selected_all, X_selected = predict_targets(
    g_val_selected,
    model,
    device=device,
    return_log_var=True,
    target_transform=target_transform,
)



labels_selected = extract_node_array_for_selected_graphs(g_val, g_val_selected, labels)

for plot_target_index, plot_target_label in iter_plot_targets("all"):

    y_true_selected_t, y_pred_selected_t, log_var_selected_t = prepare_scalar_prediction_arrays(
        y_true_selected_all,
        y_pred_selected_all,
        log_var_selected_all,
        target_index=plot_target_index,
    )

    offset = 0
    for graph_index, g in enumerate(g_val_selected):
        n = int(g.y.shape[0])

        y_true_i = y_true_selected_t[offset:offset + n]
        y_pred_i = y_pred_selected_t[offset:offset + n]
        offset += n

        mae_i = finite_mae(y_true_i, y_pred_i)
        mse_i = finite_mse(y_true_i, y_pred_i)

        result = project_predictions_to_mesh(
            graph_index,
            g_val_selected,
            y_true_selected_t,
            y_pred_selected_t,
            meta_lookup=selected_meta_lookup,
        )

        fig = plot_projected_true_vs_pred(
            result,
            title=(
                f"FORCED | {plot_target_label} | {result['organoid_str']} | "
                f"MAE={mae_i:.4f} | MSE={mse_i:.4f}"
            ),
        )
        fig.show()

In [ ]:
from src.plotting.mesh_plots import plot_marker_vs_cluster_mesh

DEFAULT_MARKER_COLORS = {
    "Lysozyme": "#2C75D2",
    "Serotonin": "#F392E3",
    "Mucin 2": "#3EC1D9",
    "Chroma": "#D852CB",
    "Glucagon": "#983EE2",
    "LGR5": "#FFB431",
    "AldoB": "#F16C6A",
    "Agr2": "#359BD5",
    "KI67": "#808080",
    "none": "#EBEBEB",
}

for gi in range(len(g_val_selected)):
    fig = plot_marker_vs_cluster_mesh(
        gi,
        g_val_selected,
        cluster_labels_all=labels_selected,
        marker_names=marker_names,
        meta_lookup=val_meta_lookup,
        marker_colors=DEFAULT_MARKER_COLORS,
    )
    fig.show()
